In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans


In [2]:
monthly_df = pd.read_csv("../data/processed/monthly_pin_raw.csv")
pin_summary_df = pd.read_csv("../data/processed/pin_summary.csv")

print(monthly_df.shape)
print(pin_summary_df.shape)

monthly_df.head()


(982, 11)
(96, 18)


,state,district,pincode,month,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_
0,Maharashtra,Thane,400601,2025-03,0.0,0.0,0.0,99.0,1058.0,0.0,0.0
1,Maharashtra,Thane,400601,2025-04,0.0,0.0,0.0,104.0,783.0,393.0,831.0
2,Maharashtra,Thane,400601,2025-05,80.0,30.0,14.0,0.0,0.0,575.0,1122.0
3,Maharashtra,Thane,400601,2025-06,53.0,33.0,1.0,0.0,0.0,165.0,670.0
4,Maharashtra,Thane,400601,2025-07,0.0,0.0,0.0,0.0,0.0,407.0,912.0


🔹 CELL 3 — Create monthly totals (Point 5 base)

In [5]:
monthly_df["total_updates"] = (
    monthly_df["demo_age_5_17"] +
    monthly_df["demo_age_17_"] +
    monthly_df["bio_age_5_17"] +
    monthly_df["bio_age_17_"]
)

monthly_df["total_enrolments"] = (
    monthly_df["age_0_5"] +
    monthly_df["age_5_17"] +
    monthly_df["age_18_greater"]
)


In [6]:
monthly_df[["pincode", "total_updates", "total_enrolments"]].head()


,pincode,total_updates,total_enrolments
0,400601,1157.0,0.0
1,400601,2111.0,0.0
2,400601,1697.0,124.0
3,400601,835.0,87.0
4,400601,1319.0,0.0


_______________________________________________________________________________________________________________________________________

Cell 4 — Calculate BDR & CBCR (monthly level)

In [7]:
# Adult Biometric Distress Ratio (BDR)
monthly_df["BDR"] = (
    monthly_df["bio_age_17_"] /
    (monthly_df["demo_age_17_"] + 1)
)

# Child Biometric Compliance Ratio (CBCR)
monthly_df["CBCR"] = (
    monthly_df["bio_age_5_17"] /
    (monthly_df["bio_age_5_17"] + monthly_df["demo_age_5_17"] + 1)
)


In [8]:
monthly_df[["pincode", "BDR", "CBCR"]].head()


,pincode,BDR,CBCR
0,400601,0.000000,0.000000
1,400601,1.059949,0.789157
2,400601,1122.000000,0.998264
3,400601,670.000000,0.993976
4,400601,912.000000,0.997549


____________________________________________________________________________________________________________________________________________

Cell 5 — Volatility (month-to-month instability)

In [9]:
# Sort to ensure correct month order
monthly_df = monthly_df.sort_values(["pincode", "month"])

# Month-to-month percentage change in updates
monthly_df["update_pct_change"] = (
    monthly_df
    .groupby("pincode")["total_updates"]
    .pct_change()
)


In [10]:
monthly_df[["pincode", "total_updates", "update_pct_change"]].head(10)


,pincode,total_updates,update_pct_change
0,400601,1157.0,NaN
1,400601,2111.0,0.824546
2,400601,1697.0,-0.196116
3,400601,835.0,-0.507955
4,400601,1319.0,0.579641
5,400601,941.0,-0.286581
6,400601,2593.0,1.755579
7,400601,1910.0,-0.263401
8,400601,4551.0,1.382723
9,400601,4321.0,-0.050538


___________________________________________________________________________________________________________________________________________

Cell 6 — Aggregate to one row per PIN (CORE STEP)

In [11]:
pin_features = (
    monthly_df
    .groupby("pincode")
    .agg(
        avg_monthly_updates=("total_updates", "mean"),
        avg_monthly_enrolments=("total_enrolments", "mean"),
        median_BDR=("BDR", "median"),
        mean_CBCR=("CBCR", "mean"),
        update_volatility=("update_pct_change", "std"),
        months_total=("total_updates", "count")
    )
    .reset_index()
)

pin_features.head()


,pincode,avg_monthly_updates,avg_monthly_enrolments,median_BDR,mean_CBCR,update_volatility,months_total
0,400601,1976.000000,79.272727,1.494118,0.842027,0.868807,11
1,400602,574.181818,15.818182,1.327684,0.863107,0.753396,11
2,400603,698.727273,13.909091,3.009091,0.953061,1.775721,11
3,400604,3189.090909,136.181818,2.311615,0.925556,0.604123,11
4,400605,3180.909091,77.454545,1.466727,0.888354,0.561921,11


____________________________________________________________________________________________________________________________________________
Cell 7 clean NaNs & prepare for clustering

In [12]:
pin_features["update_volatility"] = pin_features["update_volatility"].fillna(0)


In [13]:
pin_features.isna().sum()


pincode                   0
avg_monthly_updates       0
avg_monthly_enrolments    0
median_BDR                0
mean_CBCR                 0
update_volatility         0
months_total              0
dtype: int64

___________________________________________________________________________________________________________________________________________
Cell 8 Scale features ( before k-means)

In [14]:
from sklearn.preprocessing import RobustScaler

feature_cols = [
    "avg_monthly_updates",
    "avg_monthly_enrolments",
    "median_BDR",
    "mean_CBCR",
    "update_volatility"
]

scaler = RobustScaler()
X_scaled = scaler.fit_transform(pin_features[feature_cols])


____________________________________________________________________________________________________________________________________________
Cell 9 — Run k-means clustering (k = 4)

In [17]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
pin_features["cluster_id"] = kmeans.fit_predict(X_scaled)

pin_features["cluster_id"].value_counts()


cluster_id
0    85
3     6
1     4
2     1
Name: count, dtype: int64

In [18]:
cluster_summary = (
    pin_features
    .groupby("cluster_id")[feature_cols]
    .median()
)

cluster_summary


,avg_monthly_updates,avg_monthly_enrolments,median_BDR,mean_CBCR,update_volatility
cluster_id,,,,,
0,698.727273,18.909091,1.642857,0.850181,1.004398
1,607.818182,11.590909,212.500000,0.940468,0.769494
2,751.909091,13.454545,453.000000,0.935348,0.510631
3,712.100000,10.409091,142.290667,0.875542,1.046462


In [19]:
cluster_label_map = {
    0: "Stable / Regular",
    1: "Biometric Distress",
    2: "Biometric Distress",   # merged
    3: "High-Churn / Migration"
}

pin_features["cluster_label"] = pin_features["cluster_id"].map(cluster_label_map)

pin_features["cluster_label"].value_counts()


cluster_label
Stable / Regular          85
High-Churn / Migration     6
Biometric Distress         5
Name: count, dtype: int64

In [20]:
pin_features.to_csv("../data/processed/pin_clusters.csv", index=False)


In [21]:
pin_features.groupby("cluster_label")[
    ["avg_monthly_updates", "avg_monthly_enrolments", "median_BDR"]
].median()


,avg_monthly_updates,avg_monthly_enrolments,median_BDR
cluster_label,,,
Biometric Distress,702.000000,12.090909,220.000000
High-Churn / Migration,712.100000,10.409091,142.290667
Stable / Regular,698.727273,18.909091,1.642857


**Outcome**

PINs were segmented into three operational archetypes: Stable / Regular, High-Churn / Migration, and Biometric Distress.

Each group shows clearly different update volumes, enrolment activity, and biometric stress, enabling UIDAI to deploy targeted 
interventions instead of uniform policies.